# TODO: after get the training job running, (1) revisit drug bank data for more drug targets -> more diseases; (2) reconsider lower bound for number of in-ppi drug targets -> fewer diseases; (3) reconsider ppi size lower bound -> fewer diseases; (4) also need to adjust hyperparameter of model

# Try training the model using my_edge_list_prep_Mar9_2025.ipynb generated ppis

## (1) use the original hyperparams in

https://github.com/mims-harvard/PINNACLE/blob/main/pinnacle/run_pinnacle.sh

In [ ]:
nepochs=250
MAX_EPOCHS=250
output_filename='try1_Mar11'

### debug.... step into train.py

In [1]:
cd /home/yang1641/PINNACLE/pinnacle

/home/yang1641/PINNACLE/pinnacle


In [2]:
# General
import numpy as np
import random
import argparse
import os
import copy

# Pytorch
import torch
import torch.nn as nn
from torch_geometric.utils.convert import to_networkx, to_scipy_sparse_matrix
from torch_geometric.data import Data
from torch_geometric.utils import negative_sampling

# Center loss
from center_loss import CenterLoss

# W&B
import wandb

# Own code
from generate_input import read_data, get_metapaths, get_centerloss_labels
import model as mdl
import utils
import minibatch_utils as mb_utils
from parse_args import get_args, get_hparams

# Seed
seed = 3
print("SEED:", seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed) 
np.random.seed(seed)
random.seed(seed)
# torch.manual_seed(seed)
torch.backends.cudnn.deterministic = True

/home/yang1641/.conda/envs/pinnacle/lib/python3.8/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


SEED: 3


In [3]:
torch.cuda.is_available()

True

In [4]:
# see https://github.com/mims-harvard/PINNACLE/blob/main/pinnacle/run_pinnacle.sh
# and https://github.com/mims-harvard/PINNACLE/blob/main/pinnacle/parse_args.py

args_manual = {
    'pc_att_channels': 16,
    'feat_mat': 1024, 
    'output': 16,
    'hidden': 64,
    'lr': 0.01,
    'wd': 5e-4,
    'dropout': 0.6,
    'gradclip': 1.0,
    'n_heads': 8,
    'lmbda': 0.1,
    'theta': 0.3,
    'lr_cent': 0.1,
    'loss_type': "BCE",
    'plot': False,
    'G_f': '/scratch/gilbreth/yang1641/exome/data/PINNACLE/my_networks/string_otar_biogrid_hippie_huri_menche_and_more_all_merged_ppi_human_proteins.txt', 
    ## stop using numerical phecode for nodes / in filenames
#     'ppi_dir': '/scratch/gilbreth/yang1641/exome/data/PINNACLE/my_networks/ppi_edgelists/', 
#     'mg_f': '/scratch/gilbreth/yang1641/exome/data/PINNACLE/my_networks/phecode_phecode_edge_list_33.txt', 
    ## use phecode description for nodes/filenames.
    ## stop using all 33 diseases because they form two connected components
#     'ppi_dir': '/scratch/gilbreth/yang1641/exome/data/PINNACLE/my_networks/ppi_edgelists_phecode_description/',
#     'mg_f': '/scratch/gilbreth/yang1641/exome/data/PINNACLE/my_networks/phecode_description_phecode_description_edge_list_33.txt',
    ## use 24 diseases that form a larger connected component
    'ppi_dir': '/scratch/gilbreth/yang1641/exome/data/PINNACLE/my_networks/ppi_edgelists_phecode_description_24/',
    'mg_f': '/scratch/gilbreth/yang1641/exome/data/PINNACLE/my_networks/phecode_description_phecode_description_edge_list_24.txt',
    'epochs': 10, 
    'resume_run': '', 
    'loader': 'graphsaint', 
    'batch_size': 8, 
    'norm': None,
    'save_prefix': '/scratch/gilbreth/yang1641/exome/results/pinnacle/my_disease_ppi_task/epoch10/debug_try1', 
    'wandb_run_id': 'debug_try1'
              }

In [5]:
mkdir -p /scratch/gilbreth/yang1641/exome/results/pinnacle/my_disease_ppi_task/epoch10/debug_try1

In [6]:
import argparse

# Convert dictionary to argparse Namespace
args = argparse.Namespace(**args_manual)

type(args)
print(args)

# Generate hyperparameters
hparams_raw = get_hparams(args)

Namespace(G_f='/scratch/gilbreth/yang1641/exome/data/PINNACLE/my_networks/string_otar_biogrid_hippie_huri_menche_and_more_all_merged_ppi_human_proteins.txt', batch_size=8, dropout=0.6, epochs=10, feat_mat=1024, gradclip=1.0, hidden=64, lmbda=0.1, loader='graphsaint', loss_type='BCE', lr=0.01, lr_cent=0.1, mg_f='/scratch/gilbreth/yang1641/exome/data/PINNACLE/my_networks/phecode_description_phecode_description_edge_list_24.txt', n_heads=8, norm=None, output=16, pc_att_channels=16, plot=False, ppi_dir='/scratch/gilbreth/yang1641/exome/data/PINNACLE/my_networks/ppi_edgelists_phecode_description_24/', resume_run='', save_prefix='/scratch/gilbreth/yang1641/exome/results/pinnacle/my_disease_ppi_task/epoch10/debug_try1', theta=0.3, wandb_run_id='debug_try1', wd=0.0005)
Hyperparameters: {'pc_att_channels': 16, 'feat_mat': 1024, 'output': 16, 'hidden': 64, 'lr': 0.01, 'wd': 0.0005, 'dropout': 0.6, 'gradclip': 1.0, 'n_heads': 8, 'lambda': 0.1, 'theta': 0.3, 'lr_cent': 0.1, 'loss_type': 'BCE', 'pl

In [7]:
save_log = args.save_prefix + "_gnn_train.log"
save_graph = args.save_prefix + "_graph.pkl"
save_model = args.save_prefix + "_model_save.pth"
save_plots = args.save_prefix + "_train_embed_plots.pdf"
save_ppi_embed = args.save_prefix + "_protein_embed.pth"
save_mg_embed = args.save_prefix + "_mg_embed.pth"
save_labels_dict = args.save_prefix + "_labels_dict.txt"



# log_f = open(save_log, "w")
log_f = open(save_log, "a")  # 'append' so we don't overwrite previous logs
print('......')
print('Writing to log_f: '+save_log)
print('......')

log_f.write("Number of epochs: %s \n" % args.epochs)
log_f.write("Save model directory: %s \n" % save_model)
log_f.write("Save embeddings directory: %s, %s \n" % (save_ppi_embed, save_mg_embed))
log_f.write("Save graph: %s \n" % save_graph)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)
if device.type == 'cuda': print(torch.cuda.get_device_name(0))
best_val_acc = -1
best_model = None
eps = 10e-4


# import wandb
wandb.login(key="f362484a2783a681fd551a00093aad3ce3139d4a")
# wandb.init(config = hparams_raw, project = "pinnacle", allow_val_change=True)
if args.wandb_run_id:
    print(f"Attaching to existing wandb run ID {args.wandb_run_id}")
    wandb.init(
        config=hparams_raw,
        project="pinnacle",
        allow_val_change=True,
        id=args.wandb_run_id,
        resume="allow"
    )
else:
    print("No wandb_run_id provided; creating a brand new run.")
    wandb.init(config=hparams_raw, project="pinnacle", allow_val_change=True)

hparams = wandb.config

......
Writing to log_f: /scratch/gilbreth/yang1641/exome/results/pinnacle/my_disease_ppi_task/epoch10/debug_try1_gnn_train.log
......
Using device: cuda
NVIDIA A100-SXM4-40GB


/home/yang1641/.conda/envs/pinnacle/lib/python3.8/site-packages/torch/cuda/__init__.py:146: UserWarning: 
NVIDIA A100-SXM4-40GB with CUDA capability sm_80 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_37 sm_50 sm_60 sm_61 sm_70 sm_75 compute_37.
If you want to use the NVIDIA A100-SXM4-40GB GPU with PyTorch, please check the instructions at https://pytorch.org/get-started/locally/

  warnings.warn(incompatible_device_warn.format(device_name, capability, " ".join(arch_list), device_name))
Failed to detect the name of this notebook, you can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: Currently logged in as: yang1641 (yang1641-purdue-university). Use `wandb login --relogin` to force relogin
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `

Attaching to existing wandb run ID debug_try1


### (1) Mar 12, 2025, 12:52: this code cleared after updating read_ppi

In [13]:
# Read data
ppi_data, mg_data, edge_attr_dict, celltype_map, tissue_neighbors, ppi_layers, metagraph = read_data(args.G_f, args.ppi_dir, args.mg_f, hparams['feat_mat'])
ppi_metapaths, mg_metapaths = get_metapaths()
center_loss_labels, train_mask, val_mask, test_mask = get_centerloss_labels(args, celltype_map, ppi_layers)

Number of PPI layers: 24 24 24 24
Number of nodes: 24 Number of edges: 90
{'Obstructive chronic bronchitis': <networkx.classes.graph.Graph object at 0x14f60c8cef70>, 'Cancer of other female genital organs': <networkx.classes.graph.Graph object at 0x14f6172c84f0>, 'Cancer of prostate': <networkx.classes.graph.Graph object at 0x14f601d30550>, 'Lymphoid leukemia': <networkx.classes.graph.Graph object at 0x14f601d3b640>, 'Cancer of other lymphoid, histiocytic tissue': <networkx.classes.graph.Graph object at 0x14f601d21d30>, 'Colorectal cancer': <networkx.classes.graph.Graph object at 0x14f601d21d90>, 'Asthma': <networkx.classes.graph.Graph object at 0x14f60c8e13d0>, 'Rheumatoid arthritis and other inflammatory polyarthropathies': <networkx.classes.graph.Graph object at 0x14f60c8e1370>, 'Non-Hodgkins lymphoma': <networkx.classes.graph.Graph object at 0x14f6c59022e0>, 'Psoriasis and related disorders': <networkx.classes.graph.Graph object at 0x14f601a6fb80>, 'Cancer of stomach': <networkx.cl

In [60]:
print(args.G_f)
print(args.ppi_dir)
print(args.mg_f)
print(hparams['feat_mat'])

/scratch/gilbreth/yang1641/exome/data/PINNACLE/my_networks/string_otar_biogrid_hippie_huri_menche_and_more_all_merged_ppi_human_proteins.txt
/scratch/gilbreth/yang1641/exome/data/PINNACLE/my_networks/ppi_edgelists_phecode_description
/scratch/gilbreth/yang1641/exome/data/PINNACLE/my_networks/phecode_description_phecode_description_edge_list_33.txt
1024


step into read_data

after using 24 diseases: continue stepping into read_data

In [127]:
import glob
from collections import Counter
import pandas as pd
import numpy as np
import random
import networkx as nx
import torch
from torch_geometric.data import Data

In [128]:
G_f = args.G_f
ppi_dir = args.ppi_dir
mg_f = args.mg_f
feat_mat_dim = hparams['feat_mat']

go through mg_mapping changes - already cutoff at the beginning - still due to read_ppi strsplit

In [140]:
mg_mapping = {n: i for i, n in enumerate(sorted(ppi_layers))}
mg_mapping

{'Asthma': 0,
 'Cancer of bronchus; lung': 1,
 'Cancer of other female genital organs': 2,
 'Cancer of other lymphoid, histiocytic tissue': 3,
 'Cancer of prostate': 4,
 'Cancer of stomach': 5,
 'Cancer of urinary organs (incl': 6,
 'Cancer within the respiratory system': 7,
 'Chronic airway obstruction': 8,
 'Chronic bronchitis': 9,
 'Colorectal cancer': 10,
 'Erythematous conditions': 11,
 'Inflammatory bowel disease and other gastroenteritis and colitis': 12,
 'Leukemia': 13,
 'Lymphoid leukemia': 14,
 'Multiple myeloma': 15,
 'Myeloproliferative disease': 16,
 'Non-Hodgkins lymphoma': 17,
 'Obstructive chronic bronchitis': 18,
 'Psoriasis': 19,
 'Psoriasis and related disorders': 20,
 'Rheumatoid arthritis': 21,
 'Rheumatoid arthritis and other inflammatory polyarthropathies': 22,
 'Skin cancer': 23}

modify read_ppi

In [142]:
glob.glob(ppi_dir + "*.txt")[11]

'/scratch/gilbreth/yang1641/exome/data/PINNACLE/my_networks/ppi_edgelists_phecode_description_24/Cancer of urinary organs (incl. kidney and bladder).txt'

In [145]:
f = glob.glob(ppi_dir + "*.txt")[11]
print(f)

/scratch/gilbreth/yang1641/exome/data/PINNACLE/my_networks/ppi_edgelists_phecode_description_24/Cancer of urinary organs (incl. kidney and bladder).txt


In [147]:
context = f.split(ppi_dir)[1].split(".")[0].replace("_edgelist", "")
print(context)

Cancer of urinary organs (incl


In [149]:
f.split(ppi_dir)[1]

'Cancer of urinary organs (incl. kidney and bladder).txt'

In [150]:
f.split(ppi_dir)[1].split(".")[0]

'Cancer of urinary organs (incl'

In [151]:
f.split(ppi_dir)[1].split(".")[:-1]

['Cancer of urinary organs (incl', ' kidney and bladder)']

In [152]:
'.'.join(f.split(ppi_dir)[1].split(".")[:-1])

'Cancer of urinary organs (incl. kidney and bladder)'

In [153]:
'.'.join(f.split(ppi_dir)[1].split(".")[:-1]).replace("_edgelist", "")

'Cancer of urinary organs (incl. kidney and bladder)'

In [129]:
try:
    G = nx.read_edgelist(G_f)
    if len(G.nodes()) == 0 :
        G = nx.read_edgelist(G_f, delimiter=',')
except Exception as e1:
    try:
        G = nx.read_edgelist(G_f, delimiter=',')
    except Exception as e2:
        raise e2

feat_mat = torch.normal(torch.zeros(len(G.nodes), feat_mat_dim), std=1)

# Read PPI layers
orig_ppi_layers, ppi_layers, ppi_train, ppi_val, ppi_test = read_ppi(ppi_dir)
print("Number of PPI layers:", len(ppi_layers), len(ppi_train), len(ppi_val), len(ppi_test))
mg_mapping = {n: i for i, n in enumerate(sorted(ppi_layers))}
# Read metagraph
metagraph = nx.read_edgelist(mg_f, data=False, delimiter = "\t", create_using=nx.DiGraph)
if len(metagraph.nodes()) == 0:
    metagraph = nx.read_edgelist(mg_f, data=False, delimiter = ",", create_using=nx.DiGraph)
assert nx.is_connected(metagraph.to_undirected())

Number of PPI layers: 24 24 24 24


In [130]:
mg_feat_mat = torch.zeros(len(metagraph.nodes), feat_mat_dim)

orig_mg = metagraph
print("Number of nodes:", len(metagraph.nodes), "Number of edges:", len(metagraph.edges))
print(ppi_layers)
mg_mapping = {n: i for i, n in enumerate(sorted(ppi_layers))}
mg_mapping.update({n: i + len(ppi_layers) for i, n in enumerate(sorted([n for n in metagraph.nodes if "BTO" in n]))})
assert len(mg_mapping) == len(metagraph.nodes), set(metagraph.nodes).difference(set(list(mg_mapping.keys())))
print(mg_mapping)

Number of nodes: 24 Number of edges: 90
{'Obstructive chronic bronchitis': <networkx.classes.graph.Graph object at 0x153fec9a41c0>, 'Cancer of other female genital organs': <networkx.classes.graph.Graph object at 0x153fec9a4940>, 'Cancer of prostate': <networkx.classes.graph.Graph object at 0x153fec9a4eb0>, 'Lymphoid leukemia': <networkx.classes.graph.Graph object at 0x153fec9a4910>, 'Cancer of other lymphoid, histiocytic tissue': <networkx.classes.graph.Graph object at 0x153fec9a4a00>, 'Colorectal cancer': <networkx.classes.graph.Graph object at 0x153fefc488e0>, 'Asthma': <networkx.classes.graph.Graph object at 0x154069b9ea90>, 'Rheumatoid arthritis and other inflammatory polyarthropathies': <networkx.classes.graph.Graph object at 0x154069b9ebe0>, 'Non-Hodgkins lymphoma': <networkx.classes.graph.Graph object at 0x1540647af610>, 'Psoriasis and related disorders': <networkx.classes.graph.Graph object at 0x153ef510bd30>, 'Cancer of stomach': <networkx.classes.graph.Graph object at 0x153e

## TODO: we don't have tissue node. need to modify read_data and subsequent code

In [131]:
# Set up Data object
mg_nodetype = [0 if "BTO" in n else 1 for n in mg_mapping] # Tissue nodes = 0, Cell-type nodes = 1, protein nodes = 2
mg_edgetype = []
for edges in metagraph.edges:
    if "BTO" in edges[0] and "BTO" in edges[1]: mg_edgetype.append(0) # tissue-tissue edge
    elif "BTO" in edges[0] and "BTO" not in edges[1]: mg_edgetype.append(1) # tissue-cell edge
    elif "BTO" not in edges[0] and "BTO" in edges[1]: mg_edgetype.append(2) # cell-tissue edge
    elif "BTO" not in edges[0] and "BTO" not in edges[1]: mg_edgetype.append(3) # cell-cell edge
    else:
        print(edges)
        raise NotImplementedError

In [132]:
tissue_neighbors = {
   mg_mapping[t]: [
    mg_mapping[n]
    for n in metagraph.to_undirected().neighbors(t)
    if n in mg_mapping
]
for t in metagraph.to_undirected().nodes
if "BTO" in t and t in mg_mapping
}

empty tissue_neighbors

In [133]:
tissue_neighbors

{}

In [134]:
metagraph = nx.relabel_nodes(metagraph, mg_mapping)
mg_mask = torch.ones(len(metagraph.edges), dtype = torch.bool) # Pass in all meta graph edges during training, validation, and test
# Get all nodes after relabeling
all_nodes = list(metagraph.nodes)

one node is not converted to integer:  'Cancer of urinary organs (incl. kidney and bladder)'

a disease name is cutoff:

 'Cancer of urinary organs (incl': 6,


In [136]:
mg_mapping

{'Asthma': 0,
 'Cancer of bronchus; lung': 1,
 'Cancer of other female genital organs': 2,
 'Cancer of other lymphoid, histiocytic tissue': 3,
 'Cancer of prostate': 4,
 'Cancer of stomach': 5,
 'Cancer of urinary organs (incl': 6,
 'Cancer within the respiratory system': 7,
 'Chronic airway obstruction': 8,
 'Chronic bronchitis': 9,
 'Colorectal cancer': 10,
 'Erythematous conditions': 11,
 'Inflammatory bowel disease and other gastroenteritis and colitis': 12,
 'Leukemia': 13,
 'Lymphoid leukemia': 14,
 'Multiple myeloma': 15,
 'Myeloproliferative disease': 16,
 'Non-Hodgkins lymphoma': 17,
 'Obstructive chronic bronchitis': 18,
 'Psoriasis': 19,
 'Psoriasis and related disorders': 20,
 'Rheumatoid arthritis': 21,
 'Rheumatoid arthritis and other inflammatory polyarthropathies': 22,
 'Skin cancer': 23}

In [137]:
len(mg_mapping)

24

In [135]:
all_nodes

[5,
 10,
 1,
 7,
 4,
 23,
 16,
 3,
 17,
 13,
 14,
 15,
 0,
 8,
 9,
 11,
 19,
 2,
 'Cancer of urinary organs (incl. kidney and bladder)',
 18,
 12,
 20,
 21,
 22]

#### removed one protein from my global ppi to avoid trouble with nx.read_edgelist(G_f)

In [87]:
import glob
from collections import Counter
import pandas as pd
import numpy as np
import random
import networkx as nx
import torch
from torch_geometric.data import Data

In [88]:
G_f = args.G_f
ppi_dir = args.ppi_dir
mg_f = args.mg_f
feat_mat_dim = hparams['feat_mat']

In [89]:
# G = nx.read_edgelist(G_f, data=False)

G = nx.read_edgelist(G_f)

In [90]:
G.number_of_nodes()

23362

(previously) one node more than my local RData, because 'BCR/ABL FUSION' would be considered as two separate proteins if do not apply double quote when generating global ppi.

In [91]:
os.chdir('/scratch/gilbreth/yang1641/exome/data/PINNACLE/my_networks')

global_ppi = pd.read_csv('string_otar_biogrid_hippie_huri_menche_and_more_all_merged_ppi_human_proteins.csv')
print(global_ppi.shape)
global_ppi.head()

protein1_and_protein2 = pd.concat([global_ppi['protein1'], global_ppi['protein2']], axis=0)
allps = protein1_and_protein2.unique()

(1758460, 4)


In [92]:
allps

array(['APEX1', 'BRD2', '3BP2', ..., 'VNN3', 'XT2', 'ZNF286B'],
      dtype=object)

In [93]:
list(G.nodes)

['APEX1',
 '3BP2',
 'BRD2',
 'CBL',
 'CD244',
 'DBNL',
 'ERBB2',
 'ERBB4',
 'FGFR1',
 'FLT3',
 'FYN',
 'GRB2',
 'HCLS1',
 'HNRPL',
 'KIT',
 'LAT',
 'LCK',
 'PLCG1',
 'PLCG2',
 'VAV',
 'VAV2',
 '5-HT7',
 'CDC42',
 '5HT1A',
 'GPR39',
 'RHOA',
 'SARNP',
 'A1BG',
 'A2M',
 'ABCC6',
 'AHSG',
 'AKAP12',
 'AKNAD1',
 'ALB',
 'AMBP',
 'ANXA7',
 'ARHGAP32',
 'C11ORF52',
 'C2CD6',
 'CABYR',
 'CATSPERE',
 'CCDC136',
 'CD80',
 'CDKN1A',
 'CIART',
 'CKAP4',
 'CRISP3',
 'CRK',
 'CRKL',
 'CSF2RB',
 'EFCAB12',
 'FAM227A',
 'FCAMR',
 'FCMR',
 'FDXR',
 'FGB',
 'FOXO1',
 'FRS2',
 'GAB2',
 'GAB3',
 'GC',
 'GDPD1',
 'GLIPR1',
 'GRB7',
 'HSP90AA1',
 'IL12RB2',
 'IL18RAP',
 'KCMF1',
 'KLF17',
 'KLK10',
 'MAPT',
 'MATN2',
 'MTX1',
 'NLRP12',
 'NLRP5',
 'NR0B1',
 'NRK',
 'P2RY8',
 'PEAK3',
 'PHF11',
 'PRDX4',
 'PRR14',
 'PSME4',
 'PTPN11',
 'RAB11FIP1',
 'RABEP2',
 'RAD51AP2',
 'RESF1',
 'REV3L',
 'RHBDD1',
 'RNF123',
 'RNF39',
 'SAMM50',
 'SAPCD1',
 'SERPINA1',
 'SETD7',
 'SHC1',
 'SHMT2',
 'SMN1',
 'SNCA',
 'S

In [94]:
set(G.nodes).difference(set(allps))

set()

In [95]:
set(allps).difference(set(G.nodes))

{'BCR/ABL FUSION'}

continue debugging read_data

In [96]:
feat_mat = torch.normal(torch.zeros(len(G.nodes), feat_mat_dim), std=1)

In [97]:
from generate_input import read_ppi

In [98]:
ppi_dir

'/scratch/gilbreth/yang1641/exome/data/PINNACLE/my_networks/ppi_edgelists_phecode_description/'

step into read_ppi

using phecode description as context name now, to avoid modifying read_data

In [99]:
orig_ppi_layers = dict()
ppi_layers = dict()
ppi_train = dict()
ppi_val = dict()
ppi_test = dict()

In [100]:
ppi_dir

'/scratch/gilbreth/yang1641/exome/data/PINNACLE/my_networks/ppi_edgelists_phecode_description/'

In [103]:
# Read PPI layers
orig_ppi_layers, ppi_layers, ppi_train, ppi_val, ppi_test = read_ppi(ppi_dir)

In [104]:
len(orig_ppi_layers)

33

In [105]:
orig_ppi_layers

{'Dementias': <networkx.classes.graph.Graph at 0x15400b353880>,
 'Leukemia': <networkx.classes.graph.Graph at 0x1540647d4df0>,
 'Rheumatoid arthritis and other inflammatory polyarthropathies': <networkx.classes.graph.Graph at 0x15400b1fe0d0>,
 'Asthma': <networkx.classes.graph.Graph at 0x154112b76610>,
 'Inflammatory bowel disease and other gastroenteritis and colitis': <networkx.classes.graph.Graph at 0x154005be71f0>,
 'Colorectal cancer': <networkx.classes.graph.Graph at 0x1540647af160>,
 'Cancer of stomach': <networkx.classes.graph.Graph at 0x1540647af5b0>,
 "Alzheimer's disease": <networkx.classes.graph.Graph at 0x1540647afeb0>,
 'Diabetes mellitus': <networkx.classes.graph.Graph at 0x1540647af2b0>,
 'Other retinal disorders': <networkx.classes.graph.Graph at 0x1540647af730>,
 'Chronic bronchitis': <networkx.classes.graph.Graph at 0x1540647af520>,
 'Cancer of other lymphoid, histiocytic tissue': <networkx.classes.graph.Graph at 0x1540647af4f0>,
 'Chronic airway obstruction': <netwo

In [106]:
ppi_layers

{'Dementias': <networkx.classes.graph.Graph at 0x15400b2c0400>,
 'Leukemia': <networkx.classes.graph.Graph at 0x1540647d4e80>,
 'Rheumatoid arthritis and other inflammatory polyarthropathies': <networkx.classes.graph.Graph at 0x15400b2c0490>,
 'Asthma': <networkx.classes.graph.Graph at 0x154005be7130>,
 'Inflammatory bowel disease and other gastroenteritis and colitis': <networkx.classes.graph.Graph at 0x1540647af760>,
 'Colorectal cancer': <networkx.classes.graph.Graph at 0x1540647af910>,
 'Cancer of stomach': <networkx.classes.graph.Graph at 0x1540647af4c0>,
 "Alzheimer's disease": <networkx.classes.graph.Graph at 0x1540647af460>,
 'Diabetes mellitus': <networkx.classes.graph.Graph at 0x1540647af1f0>,
 'Other retinal disorders': <networkx.classes.graph.Graph at 0x1540647af670>,
 'Chronic bronchitis': <networkx.classes.graph.Graph at 0x1540647af850>,
 'Cancer of other lymphoid, histiocytic tissue': <networkx.classes.graph.Graph at 0x1540647afc70>,
 'Chronic airway obstruction': <netwo

In [107]:
ppi_train

{'Dementias': tensor([ True,  True,  True,  ..., False,  True,  True]),
 'Leukemia': tensor([False,  True,  True,  ...,  True,  True, False]),
 'Rheumatoid arthritis and other inflammatory polyarthropathies': tensor([True, True, True,  ..., True, True, True]),
 'Asthma': tensor([ True,  True, False,  ...,  True,  True,  True]),
 'Inflammatory bowel disease and other gastroenteritis and colitis': tensor([ True,  True,  True,  ...,  True, False,  True]),
 'Colorectal cancer': tensor([True, True, True,  ..., True, True, True]),
 'Cancer of stomach': tensor([ True,  True, False,  ...,  True,  True,  True]),
 "Alzheimer's disease": tensor([ True, False,  True,  ...,  True, False, False]),
 'Diabetes mellitus': tensor([ True, False,  True,  ...,  True,  True,  True]),
 'Other retinal disorders': tensor([False,  True,  True,  ...,  True, False,  True]),
 'Chronic bronchitis': tensor([ True, False,  True,  ...,  True,  True,  True]),
 'Cancer of other lymphoid, histiocytic tissue': tensor([ Tr

In [108]:
for k in ppi_train.keys():
    print(ppi_train[k].shape)

torch.Size([10260])
torch.Size([42841])
torch.Size([28739])
torch.Size([27219])
torch.Size([36450])
torch.Size([1179])
torch.Size([18392])
torch.Size([39058])
torch.Size([15403])
torch.Size([5158])
torch.Size([19533])
torch.Size([17235])
torch.Size([136620])
torch.Size([28425])
torch.Size([58417])
torch.Size([30122])
torch.Size([57018])
torch.Size([31514])
torch.Size([33780])
torch.Size([74])
torch.Size([18943])
torch.Size([129470])
torch.Size([62686])
torch.Size([11265])
torch.Size([30397])
torch.Size([20294])
torch.Size([17379])
torch.Size([17449])
torch.Size([25075])
torch.Size([1234])
torch.Size([11185])
torch.Size([1308])
torch.Size([55413])


In [110]:
for k in ppi_val.keys():
    print(ppi_val[k].shape)

torch.Size([10260])
torch.Size([42841])
torch.Size([28739])
torch.Size([27219])
torch.Size([36450])
torch.Size([1179])
torch.Size([18392])
torch.Size([39058])
torch.Size([15403])
torch.Size([5158])
torch.Size([19533])
torch.Size([17235])
torch.Size([136620])
torch.Size([28425])
torch.Size([58417])
torch.Size([30122])
torch.Size([57018])
torch.Size([31514])
torch.Size([33780])
torch.Size([74])
torch.Size([18943])
torch.Size([129470])
torch.Size([62686])
torch.Size([11265])
torch.Size([30397])
torch.Size([20294])
torch.Size([17379])
torch.Size([17449])
torch.Size([25075])
torch.Size([1234])
torch.Size([11185])
torch.Size([1308])
torch.Size([55413])


In [111]:
for k in ppi_test.keys():
    print(ppi_test[k].shape)

torch.Size([10260])
torch.Size([42841])
torch.Size([28739])
torch.Size([27219])
torch.Size([36450])
torch.Size([1179])
torch.Size([18392])
torch.Size([39058])
torch.Size([15403])
torch.Size([5158])
torch.Size([19533])
torch.Size([17235])
torch.Size([136620])
torch.Size([28425])
torch.Size([58417])
torch.Size([30122])
torch.Size([57018])
torch.Size([31514])
torch.Size([33780])
torch.Size([74])
torch.Size([18943])
torch.Size([129470])
torch.Size([62686])
torch.Size([11265])
torch.Size([30397])
torch.Size([20294])
torch.Size([17379])
torch.Size([17449])
torch.Size([25075])
torch.Size([1234])
torch.Size([11185])
torch.Size([1308])
torch.Size([55413])


return to read_data

In [112]:
print("Number of PPI layers:", len(ppi_layers), len(ppi_train), len(ppi_val), len(ppi_test))
mg_mapping = {n: i for i, n in enumerate(sorted(ppi_layers))}

Number of PPI layers: 33 33 33 33


In [117]:
# Read metagraph
metagraph = nx.read_edgelist(mg_f, data=False, delimiter = "\t", create_using=nx.DiGraph)

In [118]:
list(metagraph.nodes)

['Cancer of stomach',
 'Colorectal cancer',
 'Cancer of bronchus; lung',
 'Cancer within the respiratory system',
 'Cancer of prostate',
 'Skin cancer',
 'Myeloproliferative disease',
 'Cancer of other lymphoid, histiocytic tissue',
 'Non-Hodgkins lymphoma',
 'Leukemia',
 'Lymphoid leukemia',
 'Multiple myeloma',
 'Asthma',
 'Chronic airway obstruction',
 'Chronic bronchitis',
 'Erythematous conditions',
 'Psoriasis',
 'Cancer of other female genital organs',
 'Cancer of urinary organs (incl. kidney and bladder)',
 'Obstructive chronic bronchitis',
 'Inflammatory bowel disease and other gastroenteritis and colitis',
 'Psoriasis and related disorders',
 'Rheumatoid arthritis',
 'Rheumatoid arthritis and other inflammatory polyarthropathies',
 'Diabetes mellitus',
 'Type 2 diabetes',
 'Overweight, obesity and other hyperalimentation',
 'Obesity',
 "Alzheimer's disease",
 'Delirium dementia and amnestic and other cognitive disorders',
 'Dementias',
 'Multiple sclerosis',
 'Other retinal d

In [121]:
if len(metagraph.nodes()) == 0:
    metagraph = nx.read_edgelist(mg_f, data=False, delimiter = ",", create_using=nx.DiGraph)
assert nx.is_connected(metagraph.to_undirected())

AssertionError: 